# Basic matchups with PACE data

```
import point_collocation as pc
```

* Create a plan for files to use `pc.plan()`
* Print the plan to check it `print(plan.summary())`
* Do the plan and get matchups `pc.matchup(plan, geometry="grid")`

## Read in some points

In [1]:
import pandas as pd
time = "2025-04-09"
lat = 30.0
lon = -89.0

df = pd.DataFrame(
    {
        "lat": [lat],
        "lon": [lon],
        "time": [time],
    }
)
df["time"] = pd.to_datetime(df["time"])

## Create a plan

In [32]:
%%time
import point_collocation as pc
plan = pc.plan(
    df,
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L3M_Rrs",
        "granule_name": "*.8D.*.4km.*",
    }
)

CPU times: user 18.7 ms, sys: 219 μs, total: 18.9 ms
Wall time: 16.6 s


### Look at variables in that dataset

In [3]:
plan.show_variables(geometry="grid")

Dimensions : {'lat': 4320, 'lon': 8640, 'wavelength': 172, 'rgb': 3, 'eightbitcolor': 256}
Variables  : ['Rrs', 'palette']


In [4]:
plan.summary()

Plan: 1 points → 1 unique granule(s)
  Points with 0 matches : 0
  Points with >1 matches: 0
  Variables  : []
  Time buffer: 0 days 00:00:00

First 1 point(s):
  [0] lat=30.0000, lon=-89.0000, time=2025-04-09 00:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20250407_20250414.L3m.8D.RRS.V3_1.Rrs.4km.nc


## Get the matchups

For variables with a 3rd dimension, like wavelength, all variables will be shown with `_3rd dim value`.

In [6]:
res = pc.matchup(plan, geometry="grid", variables=["Rrs"])
res

,lat,lon,time,granule_id,Rrs_346,Rrs_348,Rrs_351,Rrs_353,Rrs_356,Rrs_358,...,Rrs_706,Rrs_707,Rrs_708,Rrs_709,Rrs_711,Rrs_712,Rrs_713,Rrs_714,Rrs_717,Rrs_719
0,30.0,-89.0,2025-04-09,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,0.000306,0.000488,0.00065,0.000656,0.000726,0.000932,...,0.003598,0.003496,0.003386,0.003268,0.003138,0.003004,0.00286,0.002662,0.002098,0.001644


### Datasets with only lat, lon

In this case, just the variable appears.

In [9]:
%%time
import point_collocation as pc
plan = pc.plan(
    df,
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L3M_AVW",
        "granule_name": "*.DAY.*.4km.*",
    }
)
res = pc.matchup(plan, geometry="grid", variables=["avw"])
res

CPU times: user 8.32 ms, sys: 11 ms, total: 19.3 ms
Wall time: 9.02 s


## Plan with many files

If you are not sure what files to use, you can use a short name without `granule_name`. Then look at the plan summary to see the file names. You just need to look at one file (`n=1`). In this example, there are 16 files that match. 2 resolutions (4km and 0.1 deg) and 8 temporal resolutions:

* `R32`: rolling 32 days starting every 7 days, 4 dates
* `SNSP`: seasonal/quarterly
* `8D`: 8 day
* `DAY`: daily
* `MO`: monthly starting 1st day of each month to last

In [5]:
import point_collocation as pc
plan = pc.plan(
    df,
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L3M_AVW",
    }
)

In [6]:
plan.summary(n=1)

Plan: 1 points → 16 unique granule(s)
  Points with 0 matches : 0
  Points with >1 matches: 1
  Variables  : []
  Time buffer: 0 days 00:00:00

First 1 point(s):
  [0] lat=30.0000, lon=-89.0000, time=2025-04-09 00:00:00: 16 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20250314_20250414.L3m.R32.AVW.V3_1.avw.0p1deg.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20250314_20250414.L3m.R32.AVW.V3_1.avw.4km.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20250321_20250620.L3m.SNSP.AVW.V3_1.avw.0p1deg.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20250321_20250620.L3m.SNSP.AVW.V3_1.avw.4km.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20250322_20250422.L3m.R32.AVW.V3_1.avw.0p1deg.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20250322_20250422.L3m.R32.AVW.V3_1.avw.

### Filter to the files you want

Once you see the files names, you can filter to the ones you want. using `granule_name`. For example `*.SNSP.*.4km.*` to get the seasonal (quarterly) values. `*` are wildcard values.

In [7]:
import point_collocation as pc
plan = pc.plan(
    df,
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L3M_AVW",
        "granule_name": "*.SNSP.*.4km.*"
    }
)

In [20]:
plan.summary()

Plan: 1 points → 1 unique granule(s)
  Points with 0 matches : 0
  Points with >1 matches: 0
  Variables  : []
  Time buffer: 0 days 00:00:00

First 1 point(s):
  [0] lat=30.0000, lon=-89.0000, time=2025-04-09 00:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20250321_20250620.L3m.SNSP.AVW.V3_1.avw.4km.nc


## Try many points

In [1]:
from pathlib import Path
import earthaccess
import point_collocation as pc
import pandas as pd

HERE = Path.cwd()
POINTS_CSV = HERE / "fixtures" / "points.csv"
df_points = pd.read_csv(POINTS_CSV)  # lat, lon, date columns
print(len(df_points))
df_points.head()

595


,lat,lon,date
0,27.3835,-82.7375,2024-06-13
1,27.1190,-82.7125,2024-06-14
2,26.9435,-82.8170,2024-06-14
3,26.6875,-82.8065,2024-06-14
4,26.6675,-82.6455,2024-06-14


## Get a plan for matchups from PACE data

In [2]:
%%time
import earthaccess
import point_collocation as pc

earthaccess.login()

plan = pc.plan(
    df_points[0:100],
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L3M_AVW",
        "granule_name": "*.DAY.*.4km.*",
    }
)

CPU times: user 89.4 ms, sys: 937 μs, total: 90.3 ms
Wall time: 1.87 s


In [22]:
plan.summary(n=0)

Plan: 100 points → 131 unique granule(s)
  Points with 0 matches : 0
  Points with >1 matches: 0
  Variables  : []
  Time buffer: 0 days 00:00:00


## Get 100 matchups using that plan

In [3]:
%%time
# 739Mb
res = pc.matchup(plan[0:100], geometry="grid", variables = ["avw"], batch_size=1000)

granules 1-18 of 18 processed, 100 points matched, 00:00:12
CPU times: user 3.85 s, sys: 496 ms, total: 4.35 s
Wall time: 12.4 s


In [3]:
%%time
# 630Mb
res = pc.matchup(plan[0:100], geometry="grid", variables = ["avw"], batch_size=1)

granules 1-1 of 18 processed, 1 points matched, 00:00:04
granules 2-2 of 18 processed, 9 points matched, 00:00:04
granules 3-3 of 18 processed, 8 points matched, 00:00:05
granules 4-4 of 18 processed, 8 points matched, 00:00:05
granules 5-5 of 18 processed, 10 points matched, 00:00:06
granules 6-6 of 18 processed, 1 points matched, 00:00:06
granules 7-7 of 18 processed, 2 points matched, 00:00:06
granules 8-8 of 18 processed, 1 points matched, 00:00:07
granules 9-9 of 18 processed, 1 points matched, 00:00:07
granules 10-10 of 18 processed, 9 points matched, 00:00:08
granules 11-11 of 18 processed, 8 points matched, 00:00:08
granules 12-12 of 18 processed, 8 points matched, 00:00:09
granules 13-13 of 18 processed, 6 points matched, 00:00:09
granules 14-14 of 18 processed, 3 points matched, 00:00:10
granules 15-15 of 18 processed, 9 points matched, 00:00:10
granules 16-16 of 18 processed, 9 points matched, 00:00:11
granules 17-17 of 18 processed, 6 points matched, 00:00:11
granules 18-18

In [3]:
%%time
res = pc.matchup(plan[0:100], geometry="grid", variables = ["avw"], batch_size=1, spatial_method="xoak")

granules 1-1 of 18 processed, 1 points matched, 00:00:06
granules 2-2 of 18 processed, 9 points matched, 00:00:07
granules 3-3 of 18 processed, 8 points matched, 00:00:08
granules 4-4 of 18 processed, 8 points matched, 00:00:08
granules 5-5 of 18 processed, 10 points matched, 00:00:09
granules 6-6 of 18 processed, 1 points matched, 00:00:09
granules 7-7 of 18 processed, 2 points matched, 00:00:10
granules 8-8 of 18 processed, 1 points matched, 00:00:10
granules 9-9 of 18 processed, 1 points matched, 00:00:11
granules 10-10 of 18 processed, 9 points matched, 00:00:12
granules 11-11 of 18 processed, 8 points matched, 00:00:12
granules 12-12 of 18 processed, 8 points matched, 00:00:13
granules 13-13 of 18 processed, 6 points matched, 00:00:13
granules 14-14 of 18 processed, 3 points matched, 00:00:14
granules 15-15 of 18 processed, 9 points matched, 00:00:15
granules 16-16 of 18 processed, 9 points matched, 00:00:15
granules 17-17 of 18 processed, 6 points matched, 00:00:16
granules 18-18

In [24]:
res.head()

,lat,lon,time,granule_id,avw
0,27.3835,-82.7375,2024-06-13 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,NaN
1,27.1190,-82.7125,2024-06-14 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,NaN
2,26.9435,-82.8170,2024-06-14 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,NaN
3,26.6875,-82.8065,2024-06-14 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,NaN
4,26.6675,-82.6455,2024-06-14 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,NaN


## Try lots of products

Pick a recent data point so NRT works. Not all products have files.

In [16]:
import pandas as pd
time = "2026-01-09"
lat = 30.0
lon = -89.0

df = pd.DataFrame(
    {
        "lat": [lat],
        "lon": [lon],
        "time": [time],
    }
)
df["time"] = pd.to_datetime(df["time"])

In [7]:
import earthaccess
results = earthaccess.search_datasets(instrument="oci")

short_names = [
    item.summary()["short-name"]
    for item in results
    if "L3M" in item.summary()["short-name"]
]

print(short_names)

['PACE_OCI_L3M_UVAI_UAA_NRT', 'PACE_OCI_L3M_UVAI_UAA', 'PACE_OCI_L3M_AER_UAA_NRT', 'PACE_OCI_L3M_AER_UAA', 'PACE_OCI_L3M_AOT_NRT', 'PACE_OCI_L3M_AOT', 'PACE_OCI_L3M_AVW_NRT', 'PACE_OCI_L3M_AVW', 'PACE_OCI_L3M_CHL_NRT', 'PACE_OCI_L3M_CHL', 'PACE_OCI_L3M_CLOUD_MASK_NRT', 'PACE_OCI_L3M_CLOUD_MASK', 'PACE_OCI_L3M_CLOUD_NRT', 'PACE_OCI_L3M_CLOUD', 'PACE_OCI_L3M_KD_NRT', 'PACE_OCI_L3M_KD', 'PACE_OCI_L3M_FLH_NRT', 'PACE_OCI_L3M_FLH', 'PACE_OCI_L3M_LANDVI_NRT', 'PACE_OCI_L3M_LANDVI', 'PACE_OCI_L3M_IOP_NRT', 'PACE_OCI_L3M_IOP', 'PACE_OCI_L3M_PIC_NRT', 'PACE_OCI_L3M_PIC', 'PACE_OCI_L3M_POC_NRT', 'PACE_OCI_L3M_POC', 'PACE_OCI_L3M_PAR_NRT', 'PACE_OCI_L3M_PAR', 'PACE_OCI_L3M_CARBON', 'PACE_OCI_L3M_CARBON_NRT', 'PACE_OCI_L3M_RRS_NRT', 'PACE_OCI_L3M_RRS', 'PACE_OCI_L3M_SFREFL_NRT', 'PACE_OCI_L3M_SFREFL', 'PACE_OCI_L3M_TRGAS_NRT', 'PACE_OCI_L3M_TRGAS']


In [17]:
%%time
import point_collocation as pc
for short_name in short_names:
    print(f"\n===== {short_name} =====")
    
    try:
        plan = pc.plan(
            df,
            data_source="earthaccess",
            source_kwargs={
                "short_name": short_name,
                "granule_name":"*.DAY.*",
             }
        )
        plan.show_variables(geometry="grid")
    except Exception as e:
        print("Failed:", e)


===== PACE_OCI_L3M_UVAI_UAA_NRT =====
Failed: No granules in plan — cannot show variables.

===== PACE_OCI_L3M_UVAI_UAA =====
Failed: No granules in plan — cannot show variables.

===== PACE_OCI_L3M_AER_UAA_NRT =====
Dimensions : {'lat': 180, 'lon': 360}
Variables  : ['Aerosol_Optical_Depth_354', 'Aerosol_Optical_Depth_388', 'Aerosol_Optical_Depth_480', 'Aerosol_Optical_Depth_550', 'Aerosol_Optical_Depth_670', 'Aerosol_Optical_Depth_870', 'Aerosol_Optical_Depth_1240', 'Aerosol_Optical_Depth_2200', 'Optical_Depth_Ratio_Small_Ocean_used', 'NUV_AerosolCorrCloudOpticalDepth', 'NUV_AerosolOpticalDepthOverCloud_354', 'NUV_AerosolOpticalDepthOverCloud_388', 'NUV_AerosolOpticalDepthOverCloud_550', 'NUV_AerosolIndex', 'NUV_CloudOpticalDepth', 'AAOD_354', 'AAOD_388', 'AAOD_550']

===== PACE_OCI_L3M_AER_UAA =====
Dimensions : {'lat': 180, 'lon': 360}
Variables  : ['Aerosol_Optical_Depth_354', 'Aerosol_Optical_Depth_388', 'Aerosol_Optical_Depth_480', 'Aerosol_Optical_Depth_550', 'Aerosol_Optical_